# 02 - Retention Modeling Setup

This notebook defines the prediction task from the processed cohort: who is included, what counts as a return, and how a model should be evaluated.

Run the pipeline in the next cell, or run `python src/data_pipeline.py` from the project root in a terminal and skip that cell. Then run the remaining cells.


In [31]:
from pathlib import Path
import pandas as pd
import sys

PROJECT_DIR = Path("..").resolve()
sys.path.insert(0, str(PROJECT_DIR))


In [32]:
from src.data_pipeline import run_pipeline
run_pipeline()


Saved 124,546 users to C:\Users\ethan\Documents\projects\ecommerce-retention-analysis\data\processed\retention_cohort.parquet

Day-30 return rates:
High-intent action No: 1.91% (2,012/105,300 users); average first-3-day events: 4.8
High-intent action Yes: 1.90% (365/19,246 users); average first-3-day events: 6.9
Return-rate difference (Yes - No): -0.01 percentage points


In [33]:
DATA_PATH = PROJECT_DIR / "data" / "processed" / "retention_cohort.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError("Run src/data_pipeline.py from the project root first.")

cohort = pd.read_parquet(DATA_PATH)
cohort.head()


,user_id,first_event_time,activation_events,high_intent_actions,completed_high_intent_action,returned_day30_window
0,326029,2015-06-02 04:49:40.287000+00:00,8,0,0,0
1,761633,2015-05-18 22:36:20.965000+00:00,3,0,0,1
2,1322055,2015-06-01 21:41:54.324000+00:00,7,0,0,0
3,991664,2015-06-01 20:47:33.737000+00:00,3,0,0,0
4,1159635,2015-06-01 21:20:39.643000+00:00,3,0,0,0


## Cohort and outcome

The pipeline keeps users with 3 to 30 events in their first 3 days and enough later data to observe the full return window. The target, `returned_day30_window`, is 1 if the user has any event from day 27 up to (but not including) day 34 after their first event.

This means the model will describe this qualified cohort, not every visitor.


In [34]:
assert cohort["user_id"].is_unique, "Expected one row per user"
assert cohort["returned_day30_window"].notna().all(), "Return label has missing values"

users = len(cohort)
returners = int(cohort["returned_day30_window"].sum())
return_rate = cohort["returned_day30_window"].mean()
print(f"Eligible users: {users:,}")
print(f"Returned in days 27-33: {returners:,} ({return_rate:.1%})")


Eligible users: 124,546
Returned in days 27-33: 2,377 (1.9%)


The next notebook tests whether behavior from the first 3 days can predict `returned_day30_window` for a user.

- **Candidate features:** `activation_events` and `high_intent_actions`, both measured in the first 3 days.
- **Holdout:** reserve the latest 20% of users by `first_event_time` for testing. A time-based split checks performance on later users.
- **Metric:** report average precision and compare it with the return-rate baseline above, so performance is judged against the outcome prevalence.

The pipeline also prints return rates by high-intent action as a descriptive check. That difference can motivate testing the feature, but it does not show that the action causes a return.
